## PHASE 1 – Gold Table Validation

### Step 1: Load the Gold Table

In [0]:
gold_df = spark.table('retail_project.gold.gold_fact_sales')

#### Step 2: View the Data

Why?

- Never start modeling without seeing the data.

Look for:
- Wrong values
- Nulls
- Duplicate rows
- Data types

In [0]:
display(gold_df)

#### Step 3: Check Schema

In [0]:
gold_df.printSchema()

Why?
- Machine learning requires the correct data types.

For example:

Revenue
Should be --> double NOT in string

here, Revenve is in double only no problem.

#### Step 4: Count Rows

In [0]:
gold_df.count()

#### Step 5: Count Columns

In [0]:
gold_df.columns

In [0]:
len(gold_df.columns)

#### Step 6: Summary Statistics

In [0]:
display(gold_df.describe())

#### Step 7: Missing Values

- We'll calculate missing values for every column using PySpark.

In [0]:
from pyspark.sql.functions import col, when, count

null_df = gold_df.select([
    count(when(col(column).isNull(), column)).alias(column)
    for column in gold_df.columns
])

display(null_df)

### **Inference from Missing Values**

* **Customer details** (`customer_name`, `gender`, `city`, `state`) have **968 missing values**, indicating incomplete customer information or guest purchases.
* **Customer ID** has **133 missing values**, suggesting a few transactions are not linked to customers.
* **Product ID** has **146 missing values**, while `product_name` is complete, indicating missing product mapping.
* **Loyalty-related columns** (`loyalty_points`, `Avg_loyalty_points`) have **5,344 missing values**, showing that many customers are not part of the loyalty program or loyalty data is unavailable.
* **Revenue-related columns** (`Total_revenue`, `Avg_revenue`, `Min_price`, `Max_price`) have **3,693 missing values**, likely because these values were not generated for some records during aggregation.
* **No missing values** are found in `product_name`, `category`, `order_date`, `Year`, `Month`, `Total_quantity`, `Total_orders`, and `Avg_quantity`, indicating the core transaction data is complete.


### Step 8: Missing Value Percentage

- This is more informative than raw counts.

In [0]:
from pyspark.sql.functions import col, round, count, when

total_rows = gold_df.count()

null_percentage = gold_df.select([
    round(
        (count(when(col(column).isNull(), column)) / total_rows * 100), 2
    ).alias(column)
    for column in gold_df.columns
])

display(null_percentage)

### **Inference from Missing Value Percentage**

* **Customer details** (`customer_name`, `gender`, `city`, `state`) have **12.94%** missing values, indicating incomplete customer information.
* **Loyalty-related columns** (`loyalty_points`, `Avg_loyalty_points`) have the highest missing values (**71.44%**), suggesting most customers are not enrolled in the loyalty program or loyalty data is unavailable.
* **Revenue-related columns** (`Total_revenue`, `Avg_revenue`, `Min_price`, `Max_price`) have **49.37%** missing values, likely due to missing aggregated sales information after joins or calculations.
* **Product ID (1.95%)** and **Customer ID (1.78%)** have very few missing values, indicating good data quality for key identifiers.
* **No missing values** are found in `product_name`, `category`, `order_date`, `Year`, `Month`, `Total_quantity`, `Total_orders`, and `Avg_quantity`, showing that the core transactional data is complete and reliable.


#### Step 9: Duplicate Check

In [0]:
gold_df.count()

In [0]:
gold_df.dropDuplicates().count()

- We observe both counts are the same. So, there are no duplicate rows.

#### Step 10: Check Numeric Columns

In [0]:
numeric_columns = [
    "Total_revenue",
    "Total_quantity",
    "Total_orders",
    "Avg_revenue",
    "Avg_quantity",
    "Avg_loyalty_points",
    "Min_price",
    "Max_price",
    "loyalty_points"
]

gold_df.select(numeric_columns).describe().show()

In [0]:
# Round it to 2 decimal places for the following columns for better EDA
from pyspark.sql.functions import round, col

gold_df = gold_df \
    .withColumn("Total_revenue", round(col("Total_revenue"), 2)) \
    .withColumn("Avg_revenue", round(col("Avg_revenue"), 2)) \
    .withColumn("Min_price", round(col("Min_price"), 2)) \
    .withColumn("Max_price", round(col("Max_price"), 2))

display(gold_df)

In [0]:
# Generate summary statistics for the numeric columns
numeric_columns = [
    "Total_revenue",
    "Total_quantity",
    "Total_orders",
    "Avg_revenue",
    "Avg_quantity",
    "Avg_loyalty_points",
    "Min_price",
    "Max_price",
    "loyalty_points"
]
# Generate summary statistics
summary_df = gold_df.select(numeric_columns).describe()

# Round all numeric columns to 2 decimal places

summary_df = summary_df.select(
    col('summary'),
     *[round(col(column).cast('double'), 2).alias(column) 
       for column in numeric_columns]
)

# Display the summary statistics
display(summary_df)

### **Inference from Summary Statistics**

* **Average Revenue:** The average revenue per record is **75,901.10**, with values ranging from **1,015** to **239,685**, indicating a wide variation in sales revenue.
* **Revenue Variability:** The **standard deviation of 55,704.58** shows that revenue values are highly dispersed, suggesting significant differences in transaction values.
* **Order Quantity:** Customers purchase an average of **1.98 items** per order, with a maximum of **6 items**, indicating that most orders contain only a few products.
* **Total Orders:** The average number of orders is **1**, with a maximum of **2**, showing that most records represent a single order.
* **Loyalty Points:** Customers have an average of **1,073.69 loyalty points**, ranging from **115** to **2,000**, indicating varying levels of customer engagement in the loyalty program.
* **Product Price:** The average minimum and maximum product price is **40,647.41**, with prices ranging from **1,007** to **79,959**, showing a broad price range across products.

### **Key Insights**

* Revenue has **high variability**, indicating both low-value and high-value transactions.
* Most customers purchase **1–2 products per order**.
* The dataset contains products across **a wide price range**.
* Customer loyalty points vary significantly, reflecting different levels of customer activity.


In [0]:
# Check the schema of the DataFrame
gold_df.printSchema()

In [0]:
gold_df.describe().display()

In [0]:
# check for null values in the DataFrame Total_revenue (49% null present)
gold_df.filter(col("Total_revenue").isNull()).count()


In [0]:
# display the rows with null values in Total_revenue
gold_df.filter(col("Total_revenue").isNull()).display()

In [0]:
# Check for null values in the column customer_name
gold_df.filter(col("customer_name").isNull()).count()

In [0]:
# display the rows with null values in customer_name
gold_df.filter(col("customer_name").isNull()).display()

### Before EDA, We Need to Investigate One Thing

In [0]:
from pyspark.sql.functions import col

display(
    gold_df.filter(col("Total_revenue").isNull())
)

In [0]:
# display the rows with null values in Total_revenue
display(
    gold_df.filter(col("Total_revenue").isNotNull())
)

Compare the two outputs.

Look specifically at:

- product_id
- customer_id
- category
- Total_quantity
- Total_orders

We want to see whether the rows with missing revenue are genuinely unsold records or whether something went wrong in the Gold-layer joins or aggregations.

Should We Start EDA Now?

Not yet.

In a real company, I would not let a team begin EDA while nearly 50% of the target variable is missing.

We need to understand those rows first.

### The Next Step (Phase 1.5)

In [0]:
# display the rows with null values in Total_revenue in below column
from pyspark.sql.functions import col

gold_df.filter(col("Total_revenue").isNull()).select\
(
    "product_id",
    "customer_id",
    "category",
    "Total_quantity",
    "Total_orders",
    "Total_revenue",
    "Min_price",
    "Max_price"
).display()

In [0]:
# display the rows with not null values in Total_revenue in below column
gold_df.filter(col("Total_revenue").isNotNull()) \
       .select(
           "product_id",
           "customer_id",
           "category",
           "Total_quantity",
           "Total_orders",
           "Total_revenue"
       ).display()

Here's the most important finding.

##### 🔍 Analysis of your uploaded tables

For the rows where Total_revenue is NULL, I noticed:

- Total_quantity is NOT NULL
- Total_orders is NOT NULL
- Total_revenue is NULL
- Min_price is NULL
- Max_price is NULL

This strongly suggests that your **aggregated metrics** table didn't have matching records for these combinations during the **Gold-layer join**. In other words, this is more likely a Gold-layer data modeling issue than a machine learning issue.

#### Should we fix this now?
No.

Since your **goal is to learn the Databricks Data Science workflow**, we don't want to spend a long time redesigning the Gold layer unless it's required.

Instead, we'll create an ML-ready dataset from the Gold table.

This is exactly what many companies do: the business Gold table remains unchanged, while data scientists prepare a separate dataset for modeling.

Use the Not Null values dataset for ML Modling.

In [0]:
gold_df.write.format('delta').mode('overwrite').saveAsTable('retail_project.gold.gold_validation_fact_sales')